# Notebook 3: MobileNetV2 Model Evaluation

**Image Manipulation Detection System — Google Colab**

This notebook evaluates the trained MobileNetV2 model on the **untouched test split**.
No training, fine-tuning, or model selection occurs here.

---

> **Important:** The test dataset was held out during all training and validation.
> It is used exactly once — here — to produce unbiased performance estimates.
> Do not use test results to adjust hyperparameters or retrain the model.
> All reported metrics come from actual `model.predict()` calls on real data.
> No results are fabricated or assumed.

**Prerequisites:**
- `01_dataset_preparation.ipynb` completed — `data/metadata/dataset_split.csv` exists
- `02_mobilenetv2_training.ipynb` completed — `artifacts/models/mobilenetv2.keras` exists

**Outputs produced:**
- `artifacts/models/evaluation_results.json`
- `artifacts/plots/confusion_matrix.png`
- `artifacts/plots/evaluation_results.csv`

## Step 1: Clone Repository and Install Dependencies

In [ ]:
# TODO: Replace with your actual GitHub repository URL and repo folder name
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
REPO_NAME = "YOUR_REPO"

import os

if not os.path.exists(REPO_NAME):
    print(f"Cloning {GITHUB_REPO_URL} ...")
    os.system(f"git clone {GITHUB_REPO_URL}")
else:
    print(f"Repository folder '{REPO_NAME}' exists — pulling latest changes.")
    os.system(f"cd {REPO_NAME} && git pull")

os.chdir(REPO_NAME)
print("Working directory:", os.getcwd())

In [ ]:
import subprocess
result = subprocess.run(
    ["pip", "install", "-r", "requirements.txt", "-q"],
    capture_output=True, text=True
)
print(result.stdout[-300:] if result.stdout else "Installation complete.")
if result.returncode != 0:
    print("ERRORS:", result.stderr[-300:])

## Step 2: GPU Check and Imports

In [ ]:
import sys
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from sklearn.metrics import classification_report

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("Available GPUs:", gpus if gpus else "None (CPU mode)")

# Add project root to Python path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import Config
from src.evaluation.metrics import compute_classification_metrics, compute_confusion_matrix
from src.training.train_mobilenet import build_dataset_frame
from src.preprocessing.image_normalizer import normalize_image

print("\nProject modules imported successfully.")

## Step 3: Configure Paths

In [ ]:
SPLIT_CSV_PATH = Config.DATASET_SPLIT_PATH
MODEL_DIR      = Config.MODEL_DIR
PLOTS_DIR      = Config.PLOTS_DIR

# Prefer Colab-export filename; fall back to legacy name
MODEL_PATH = MODEL_DIR / "mobilenetv2.keras"
if not MODEL_PATH.exists():
    MODEL_PATH = Config.MODEL_PATH_MOBILENET  # mobilenet_model.keras

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Split CSV   :", SPLIT_CSV_PATH)
print("Split exists:", SPLIT_CSV_PATH.exists())
print("Model path  :", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())

if not SPLIT_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Dataset split CSV not found at {SPLIT_CSV_PATH}.\n"
        "Please run 01_dataset_preparation.ipynb first."
    )
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Trained model not found at {MODEL_PATH}.\n"
        "Please run 02_mobilenetv2_training.ipynb first."
    )

## Step 4: Load the Trained Model

In [ ]:
print(f"Loading model from {MODEL_PATH} ...")
model = tf.keras.models.load_model(str(MODEL_PATH))
model_size_bytes = MODEL_PATH.stat().st_size
model_size_mb = model_size_bytes / (1024 ** 2)

print(f"Model loaded successfully.")
print(f"  Model file : {MODEL_PATH.name}")
print(f"  File size  : {model_size_mb:.1f} MB")
print(f"  Input shape: {Config.INPUT_SHAPE}")

# Warmup pass to build computation graph before timing inference
warmup = tf.zeros((1,) + Config.INPUT_SHAPE, dtype=tf.float32)
_ = model(warmup, training=False)
print("Warmup pass completed.")

## Step 5: Load the Test Split

The test split was held out during all of Notebooks 1 and 2.
It is loaded here using the same `dataset_split.csv` to guarantee
that exactly the same images are evaluated.

In [ ]:
print("Loading test split from dataset_split.csv ...")
try:
    test_images, test_labels = build_dataset_frame(SPLIT_CSV_PATH, split_name="test")
    print(f"Test images loaded: {len(test_images)}")
    print(f"  Shape per image : {test_images[0].shape}")
    print(f"  Label distribution:")
    unique, counts = np.unique(test_labels, return_counts=True)
    for label, count in zip(unique, counts):
        label_name = Config.CLASS_LABELS.get(label, str(label))
        print(f"    {label_name} (label={label}): {count}")
except (FileNotFoundError, ValueError) as e:
    raise RuntimeError(
        f"Failed to load test data: {e}\n"
        "Ensure dataset_split.csv is present and all test image paths are reachable."
    ) from e

## Step 6: Run Inference on the Test Set

Predictions are computed in batches. Inference time is recorded per image
to measure real-world latency.

In [ ]:
print(f"Running inference on {len(test_images)} test images ...")

# Build a tf.data.Dataset for the test set (batched, no shuffle)
test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels))
test_dataset = test_dataset.batch(Config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

all_probabilities = []
inference_times_ms = []

for batch_images, _ in test_dataset:
    t_start = time.perf_counter()
    batch_probs = model.predict(batch_images, verbose=0)
    t_end = time.perf_counter()
    batch_ms = (t_end - t_start) * 1000.0
    per_image_ms = batch_ms / len(batch_images)
    inference_times_ms.extend([per_image_ms] * len(batch_images))
    all_probabilities.extend(batch_probs.flatten().tolist())

y_pred_prob = np.array(all_probabilities, dtype=np.float64)
y_true      = np.array(test_labels, dtype=np.int64)

print(f"Inference complete.")
print(f"  Total images  : {len(y_pred_prob)}")
print(f"  Prob range    : [{y_pred_prob.min():.4f}, {y_pred_prob.max():.4f}]")

## Step 7: Compute Classification Metrics

Metrics are computed using the existing `src/evaluation/metrics.py` module.
All values come from actual predictions — none are fabricated or estimated.

In [ ]:
metrics = compute_classification_metrics(
    y_true, y_pred_prob,
    threshold=Config.CLASSIFICATION_THRESHOLD
)
cm_dict = compute_confusion_matrix(
    y_true, y_pred_prob,
    threshold=Config.CLASSIFICATION_THRESHOLD
)

print("=" * 45)
print("Classification Metrics (Test Set)")
print("=" * 45)
print(f"  Accuracy   : {metrics['accuracy']:.4f}  ({metrics['accuracy']*100:.2f}%)")
print(f"  Precision  : {metrics['precision']:.4f}")
print(f"  Recall     : {metrics['recall']:.4f}")
print(f"  F1-Score   : {metrics['f1_score']:.4f}")
print(f"  ROC-AUC    : {metrics['roc_auc']:.4f}")
print(f"  Specificity: {metrics['specificity']:.4f}")
print()
print("Confusion Matrix:")
print(f"  TN={cm_dict['tn']}  FP={cm_dict['fp']}")
print(f"  FN={cm_dict['fn']}  TP={cm_dict['tp']}")

## Step 8: sklearn Classification Report

In [ ]:
y_pred_labels = (y_pred_prob >= Config.CLASSIFICATION_THRESHOLD).astype(int)
target_names  = [Config.CLASS_LABELS[0], Config.CLASS_LABELS[1]]

print("Classification Report:")
print(classification_report(y_true, y_pred_labels, target_names=target_names, digits=4))

## Step 9: Confusion Matrix Visualization

In [ ]:
cm_array = np.array([
    [cm_dict["tn"], cm_dict["fp"]],
    [cm_dict["fn"], cm_dict["tp"]],
], dtype=int)

class_names = [Config.CLASS_LABELS[0], Config.CLASS_LABELS[1]]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm_array,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
    linewidths=0.5,
    linecolor="gray",
)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
ax.set_title("Confusion Matrix — MobileNetV2 (Test Set)", fontsize=13, pad=12)
plt.tight_layout()

cm_path = PLOTS_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=Config.PLOT_DPI, bbox_inches="tight")
plt.show()
print(f"Saved: {cm_path}")

## Step 10: Training History Plots (if CSV available)

If the training history CSVs from Notebook 2 are present, the accuracy and loss
curves are plotted here for reference.

In [ ]:
phase1_csv = PLOTS_DIR / "phase1_training_history.csv"
phase2_csv = PLOTS_DIR / "phase2_training_history.csv"

histories = []
if phase1_csv.exists():
    histories.append(pd.read_csv(phase1_csv))
if phase2_csv.exists():
    histories.append(pd.read_csv(phase2_csv))

if histories:
    combined = pd.concat(histories, ignore_index=True)
    combined["epoch_overall"] = range(1, len(combined) + 1)
    phase1_len = len(histories[0]) if len(histories) > 1 else None

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for col, label, axis in [
        ("accuracy", "Train", axes[0]),
        ("val_accuracy", "Validation", axes[0]),
        ("loss", "Train", axes[1]),
        ("val_loss", "Validation", axes[1]),
    ]:
        if col in combined.columns:
            axis.plot(combined["epoch_overall"], combined[col], label=label)

    for ax, title, ylabel in [
        (axes[0], "Accuracy", "Accuracy"),
        (axes[1], "Loss", "Binary Cross-Entropy"),
    ]:
        if phase1_len and len(histories) > 1:
            ax.axvline(x=phase1_len, color="gray", linestyle="--", alpha=0.6, label="Fine-tune start")
        ax.set_title(f"Model {title}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(alpha=0.3)

    plt.suptitle("MobileNetV2 Training History (from Notebook 2)", fontsize=13)
    plt.tight_layout()
    hist_path = PLOTS_DIR / "training_history_eval.png"
    plt.savefig(hist_path, dpi=Config.PLOT_DPI, bbox_inches="tight")
    plt.show()
    print(f"Saved: {hist_path}")
else:
    print("No training history CSVs found — skipping plots.")
    print("Run Notebook 2 first to generate them.")

## Step 11: Inference Time Summary

Per-image inference time measured during batch prediction.

In [ ]:
times = np.array(inference_times_ms)

print("Inference Time (per image):")
print(f"  Mean  : {times.mean():.2f} ms")
print(f"  Std   : {times.std():.2f} ms")
print(f"  Min   : {times.min():.2f} ms")
print(f"  Max   : {times.max():.2f} ms")
print(f"  Median: {np.median(times):.2f} ms")
print()
print(f"Model file size: {model_size_mb:.1f} MB")
print(f"GPU available  : {len(tf.config.list_physical_devices('GPU')) > 0}")

## Step 12: Save Evaluation Results

> **Disclaimer:** These results are computed on the actual held-out test dataset.
> Do not use test results to adjust hyperparameters or retrain the model.
> The test set must remain untouched across all training iterations.

In [ ]:
timestamp = datetime.now(timezone.utc).isoformat()

evaluation_results = {
    "status": "evaluated",
    "evaluated_at": timestamp,
    "model_file": MODEL_PATH.name,
    "model_size_mb": round(model_size_mb, 2),
    "test_set_size": int(len(y_true)),
    "classification_threshold": Config.CLASSIFICATION_THRESHOLD,
    "metrics": {
        "accuracy"   : round(metrics["accuracy"],    4),
        "precision"  : round(metrics["precision"],   4),
        "recall"     : round(metrics["recall"],      4),
        "f1_score"   : round(metrics["f1_score"],    4),
        "roc_auc"    : round(metrics["roc_auc"],     4),
        "specificity": round(metrics["specificity"], 4),
    },
    "confusion_matrix": cm_dict,
    "inference_time_ms": {
        "mean"  : round(float(times.mean()),   2),
        "std"   : round(float(times.std()),    2),
        "min"   : round(float(times.min()),    2),
        "max"   : round(float(times.max()),    2),
        "median": round(float(np.median(times)), 2),
    },
    "tensorflow_version": tf.__version__,
    "gpu_used": len(tf.config.list_physical_devices("GPU")) > 0,
    "note": (
        "Results are from actual model.predict() calls on the held-out test split. "
        "Not fabricated or assumed."
    ),
}

# Save JSON
results_json_path = MODEL_DIR / "evaluation_results.json"
with open(results_json_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, indent=2)
print(f"Saved: {results_json_path}")

# Save CSV summary
results_csv_path = PLOTS_DIR / "evaluation_results.csv"
results_flat = {
    "model_file": MODEL_PATH.name,
    "test_set_size": len(y_true),
    "model_size_mb": round(model_size_mb, 2),
    **{f"metric_{k}": v for k, v in evaluation_results["metrics"].items()},
    "cm_tn": cm_dict["tn"],
    "cm_fp": cm_dict["fp"],
    "cm_fn": cm_dict["fn"],
    "cm_tp": cm_dict["tp"],
    "mean_inference_ms": round(float(times.mean()), 2),
    "evaluated_at": timestamp,
}
pd.DataFrame([results_flat]).to_csv(results_csv_path, index=False)
print(f"Saved: {results_csv_path}")
print("\nEvaluation complete.")

## Summary

| File | Description |
|------|-------------|
| `artifacts/models/evaluation_results.json` | Full metrics in JSON format |
| `artifacts/plots/evaluation_results.csv` | Flat metrics summary in CSV |
| `artifacts/plots/confusion_matrix.png` | Confusion matrix heatmap |
| `artifacts/plots/training_history_eval.png` | Training curves (if available) |

---

> **ELA Disclaimer:** This model was trained to detect potential image manipulation
> based on statistical patterns in the CASIA dataset. The ELA visualization shown
> by the Flask application is an investigative aid. Bright regions indicate
> potential compression inconsistencies and **do not constitute definitive proof
> of digital manipulation.**

---

**Next step:** Download `evaluation_results.json` and place it in `artifacts/models/`
in your local project. Then follow `scripts/download_model_instructions.md`
to deploy `mobilenetv2.keras` to the Flask application.